# Faithfulness de las respuestas RAFT con tres jueces

Evalúa **solo faithfulness** de las tres combinaciones del modelo RAFT (`raft-llama3.1-8b` con prompt `raft`, `estructurado` y `directo`, 196 preguntas cada una) con tres modelos juez y promedia sus puntajes.

| Juez | Modelo en OpenRouter | De dónde sale el puntaje |
|---|---|---|
| GPT | `openai/gpt-4.1-mini` | **Ya evaluado**: se lee de `evaluacion_generacion_detalle_raft.csv`, no se vuelve a llamar |
| Llama | `meta-llama/llama-3.3-70b-instruct` | Se evalúa aquí |
| DeepSeek | `deepseek/deepseek-v3.2` | Se evalúa aquí |

**No modifica nada existente.** Solo lee `resultados_generacion_llm_raft.json` y `evaluacion_generacion_detalle_raft.csv`. Escribe únicamente en carpetas nuevas:
- `validacion/checkpoints_raft_jueces/`: un CSV por juez y combinación (se reutiliza si la corrida se interrumpe).
- `validacion/reporte_raft_jueces/`: el Excel final y sus CSV.

**Sobre los jueces nuevos.** No se usan modelos de respaldo (no se envía `models` en la petición), así que cada puntaje corresponde a un solo juez. Los puntajes de GPT vienen de una corrida anterior que sí permitía respaldo.

**Advertencia de sesgo.** `llama-3.3-70b-instruct` generó los pares pregunta–respuesta del dataset y el modelo evaluado es de la familia Llama. Por eso la hoja de acuerdo entre jueces importa: si Llama puntúa distinto que los otros dos, esa puede ser la razón.

Requiere la variable de entorno `OPENROUTER_API_KEY` (o un `.env`) solo si faltan checkpoints. Ejecutar desde la carpeta del proyecto.

In [ ]:
%pip install -q ragas langchain-openai pandas openpyxl python-dotenv scipy

In [ ]:
import json
import os
import sys
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# Mismo parche del notebook original: un import roto de langchain_community que ragas necesita
if "langchain_community.chat_models.vertexai" not in sys.modules:
    try:
        import langchain_community.chat_models.vertexai  # noqa: F401
    except ModuleNotFoundError:
        _falso = types.ModuleType("langchain_community.chat_models.vertexai")

        class ChatVertexAI:  # nunca se instancia
            pass

        _falso.ChatVertexAI = ChatVertexAI
        sys.modules["langchain_community.chat_models.vertexai"] = _falso
    except ImportError:
        pass  # sin langchain_community no hace falta el parche hasta que se evalúe

## 1. Configuración

In [ ]:
# Entradas (solo lectura)
RUTA_RESPUESTAS = Path("./validacion/resultados_generacion_llm_raft.json")
RUTA_DETALLE_GPT = Path("./validacion/evaluacion_generacion_detalle_raft.csv")

# Salidas (carpetas nuevas)
DIR_CHECKPOINTS = Path("./validacion/checkpoints_raft_jueces")
DIR_SALIDA = Path("./validacion/reporte_raft_jueces")
DIR_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

# Jueces. "gpt" se reutiliza; los otros dos se evalúan en este notebook.
JUECES = {
    "gpt-4.1-mini": {"id": "openai/gpt-4.1-mini", "reutilizar": True},
    "llama-3.3-70b": {"id": "meta-llama/llama-3.3-70b-instruct", "reutilizar": False},
    "deepseek-v3.2": {"id": "deepseek/deepseek-v3.2", "reutilizar": False},
}
JUECES_NUEVOS = [j for j, c in JUECES.items() if not c["reutilizar"]]
NOMBRES_JUECES = list(JUECES)

ORDEN_PROMPTS = ["raft", "estructurado", "directo"]

# None = las 196 preguntas. Para una prueba rápida pon un número chico (ej. 5);
# los checkpoints de la muestra llevan otro nombre y no se mezclan con los completos.
TAMANO_MUESTRA = None

## 2. Respuestas a evaluar y puntajes de GPT ya existentes

Se arma una tabla con una fila por combinación y pregunta. Los puntajes de GPT se emparejan por `(combinación, pregunta)` y se verifica que la respuesta del CSV sea idéntica a la del JSON, para asegurar que se evalúa el mismo texto.

In [ ]:
with open(RUTA_RESPUESTAS, "r", encoding="utf-8") as f:
    generacion = json.load(f)

filas = []
for c in generacion["combinaciones"]:
    if not c["modelo_llm"].startswith("raft"):
        continue
    etiqueta = f"{c['pipeline_retrieve']}__{c['modelo_llm']}__{c['estilo_prompt']}"
    for r in c["resultados"]:
        filas.append({
            "combinacion": etiqueta,
            "estilo_prompt": c["estilo_prompt"],
            "id": r["id"],
            "pregunta": r["pregunta"],
            "response": r.get("respuesta_generada") or "",
            "contextos": r.get("contextos_usados") or [],
            "reference": r.get("respuesta_esperada") or "",
        })
base = pd.DataFrame(filas)
print(base.groupby(["combinacion"]).size())
assert base["estilo_prompt"].nunique() == 3, "Se esperaban los 3 prompts RAFT"
assert base.groupby("combinacion")["id"].nunique().eq(196).all(), "Cada combinación debe tener 196 ids distintos"

gpt = pd.read_csv(RUTA_DETALLE_GPT, usecols=["combinacion", "id", "user_input", "response", "faithfulness"])
gpt = gpt.rename(columns={"faithfulness": "faithfulness_gpt-4.1-mini"})
assert len(gpt) == len(base), f"El CSV de GPT tiene {len(gpt)} filas y se esperaban {len(base)}"

base = base.merge(gpt[["combinacion", "id", "response", "faithfulness_gpt-4.1-mini"]],
                  on=["combinacion", "id"], how="left", suffixes=("", "_csv"))
assert base["faithfulness_gpt-4.1-mini"].notna().sum() > 0, "No se pudo emparejar ningún puntaje de GPT"
iguales = (base["response"].fillna("").str.strip() == base["response_csv"].fillna("").str.strip())
print(f"Respuestas idénticas entre el JSON y el CSV de GPT: {int(iguales.sum())} de {len(base)}")
assert iguales.all(), "Hay respuestas distintas entre el JSON y el CSV de GPT: GPT habría evaluado otro texto"
base = base.drop(columns=["response_csv"])
print("Puntajes de GPT sin valor (NaN):", int(base["faithfulness_gpt-4.1-mini"].isna().sum()))

## 3. Evaluar con los jueces nuevos (solo faithfulness)

Cada juez y combinación se guarda en su propio CSV dentro de `checkpoints_raft_jueces/`. Si el checkpoint existe, se reutiliza; así se puede interrumpir y retomar. Las herramientas de ragas y la clave de API solo se cargan cuando falta algún checkpoint.

In [ ]:
def crear_juez(modelo_openrouter):
    """Crea el juez para ragas. No se pasa 'models': no hay modelos de respaldo."""
    from langchain_openai import ChatOpenAI
    from ragas.llms import LangchainLLMWrapper
    from ragas.metrics import Faithfulness

    clave = os.environ.get("OPENROUTER_API_KEY")
    if not clave:
        try:
            from dotenv import load_dotenv
            load_dotenv("./.env")
            clave = os.environ.get("OPENROUTER_API_KEY")
        except ImportError:
            pass
    if not clave:
        raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")

    llm = ChatOpenAI(model=modelo_openrouter, base_url="https://openrouter.ai/api/v1", api_key=clave, temperature=0)
    envuelto = LangchainLLMWrapper(llm)
    return envuelto, Faithfulness(llm=envuelto)


def evaluar_juez_combinacion(nombre_juez, combinacion, datos, intentos_externos=3, espera=60):
    sufijo = f"__muestra{TAMANO_MUESTRA}" if TAMANO_MUESTRA else ""
    ruta = DIR_CHECKPOINTS / f"{nombre_juez}__{combinacion}{sufijo}.csv"
    if ruta.exists():
        print(f"  [{nombre_juez} | {combinacion}] checkpoint reutilizado.")
        return pd.read_csv(ruta)

    from ragas import evaluate, EvaluationDataset
    from ragas.run_config import RunConfig

    llm, metrica = crear_juez(JUECES[nombre_juez]["id"])
    dataset = EvaluationDataset.from_list([
        {"user_input": r.pregunta, "response": r.response, "retrieved_contexts": r.contextos, "reference": r.reference}
        for r in datos.itertuples()
    ])
    for intento in range(1, intentos_externos + 1):
        try:
            res = evaluate(dataset=dataset, metrics=[metrica], llm=llm,
                           run_config=RunConfig(timeout=120, max_retries=8, max_wait=90, max_workers=4),
                           raise_exceptions=False, show_progress=True)
            df = res.to_pandas()
            # ragas conserva el orden: se asigna el id por posición y se comprueba la pregunta
            assert len(df) == len(datos)
            assert (df["user_input"].values == datos["pregunta"].values).all(), "El orden de ragas cambió"
            df["id"] = datos["id"].values
            df = df[["id", "user_input", "faithfulness"]]
            df.to_csv(ruta, index=False, encoding="utf-8")
            print(f"  [{nombre_juez} | {combinacion}] listo: {ruta.name}")
            return df
        except Exception as e:
            print(f"  [{nombre_juez} | {combinacion}] intento {intento}/{intentos_externos} falló: {e}")
            if intento < intentos_externos:
                time.sleep(espera)
    raise RuntimeError(f"Se agotaron los reintentos para {nombre_juez} | {combinacion}")

In [ ]:
evaluar = base if not TAMANO_MUESTRA else base.groupby("combinacion").head(TAMANO_MUESTRA)

for nombre_juez in JUECES_NUEVOS:
    print(f"\n=== Juez: {nombre_juez} ({JUECES[nombre_juez]['id']}) ===")
    partes = []
    for combinacion, datos in evaluar.groupby("combinacion", sort=False):
        df = evaluar_juez_combinacion(nombre_juez, combinacion, datos.reset_index(drop=True))
        df = df[["id", "faithfulness"]].rename(columns={"faithfulness": f"faithfulness_{nombre_juez}"})
        df["combinacion"] = combinacion
        partes.append(df)
    puntajes = pd.concat(partes, ignore_index=True)
    base = base.merge(puntajes, on=["combinacion", "id"], how="left")

print("\nPuntajes sin valor (NaN) por juez:")
print(base[[f"faithfulness_{j}" for j in NOMBRES_JUECES]].isna().sum())

## 4. Detalle: todos los puntajes y el promedio por pregunta

`promedio_jueces` es el promedio de los puntajes disponibles de esa pregunta; `n_jueces` dice cuántos jueces aportaron valor (si un juez no devolvió puntaje, no cuenta).

In [ ]:
cols_juez = [f"faithfulness_{j}" for j in NOMBRES_JUECES]
detalle = base.copy()
detalle["promedio_jueces"] = detalle[cols_juez].mean(axis=1, skipna=True)
detalle["n_jueces"] = detalle[cols_juez].notna().sum(axis=1)
detalle["estilo_prompt"] = pd.Categorical(detalle["estilo_prompt"], categories=ORDEN_PROMPTS, ordered=True)
detalle = detalle.sort_values(["estilo_prompt", "id"]).reset_index(drop=True)
detalle_xlsx = detalle[["combinacion", "estilo_prompt", "id", "pregunta"] + cols_juez + ["promedio_jueces", "n_jueces"]].copy()
detalle_xlsx["estilo_prompt"] = detalle_xlsx["estilo_prompt"].astype(str)
detalle_xlsx.head(10)

## 5. Resumen por prompt

Una fila por prompt: el promedio de cada juez (sobre las 196 preguntas) y el **promedio de jueces**, que es el promedio de esos tres promedios. También se reporta el promedio de los promedios por pregunta como verificación; solo difieren si algún puntaje falta.

In [ ]:
filas = []
for estilo in ORDEN_PROMPTS:
    sub = detalle[detalle["estilo_prompt"] == estilo]
    fila = {"estilo_prompt": estilo, "combinacion": sub["combinacion"].iloc[0], "n_preguntas": len(sub)}
    for j in NOMBRES_JUECES:
        fila[f"faithfulness_{j}"] = sub[f"faithfulness_{j}"].mean()
    fila["promedio_jueces"] = np.mean([fila[f"faithfulness_{j}"] for j in NOMBRES_JUECES])
    fila["promedio_por_pregunta"] = sub["promedio_jueces"].mean()
    fila["min_entre_jueces"] = min(fila[f"faithfulness_{j}"] for j in NOMBRES_JUECES)
    fila["max_entre_jueces"] = max(fila[f"faithfulness_{j}"] for j in NOMBRES_JUECES)
    for j in NOMBRES_JUECES:
        fila[f"nulos_{j}"] = int(sub[f"faithfulness_{j}"].isna().sum())
    filas.append(fila)
resumen = pd.DataFrame(filas)
resumen["puesto"] = resumen["promedio_jueces"].rank(ascending=False, method="min").astype(int)
resumen.round(4)

## 6. Acuerdo entre jueces

- **Puestos por juez:** si los tres jueces ordenan los prompts igual, la conclusión no depende del juez.
- **Correlación por pregunta** entre cada par de jueces (Spearman, sobre las 588 respuestas con ambos puntajes). Una correlación baja entre Llama y los otros dos apuntaría al sesgo descrito arriba.

In [ ]:
puestos = pd.DataFrame({j: resumen.set_index("estilo_prompt")[f"faithfulness_{j}"].rank(ascending=False, method="min").astype(int)
                        for j in NOMBRES_JUECES})
puestos["promedio_jueces"] = resumen.set_index("estilo_prompt")["puesto"]
display(puestos)

filas = []
for i, a in enumerate(NOMBRES_JUECES):
    for b in NOMBRES_JUECES[i + 1:]:
        par = detalle[[f"faithfulness_{a}", f"faithfulness_{b}"]].dropna()
        rho, p = stats.spearmanr(par.iloc[:, 0], par.iloc[:, 1])
        filas.append({"juez_A": a, "juez_B": b, "n": len(par), "spearman": rho, "p": p,
                      "diferencia_media (A - B)": (par.iloc[:, 0] - par.iloc[:, 1]).mean()})
acuerdo = pd.DataFrame(filas)
display(acuerdo.round(4))

## 7. Exportar (archivos nuevos)

In [ ]:
notas = pd.DataFrame({"Concepto": [
    "Métrica", "Respuestas evaluadas", "Jueces", "Puntajes de GPT", "Jueces nuevos",
    "Promedio de jueces (resumen)", "Promedio de jueces (detalle)", "Origen de las respuestas"],
    "Detalle": [
    "faithfulness (RAGAS)",
    "3 combinaciones del modelo RAFT x 196 preguntas = 588 respuestas",
    ", ".join(f"{j} ({JUECES[j]['id']})" for j in NOMBRES_JUECES),
    "Reutilizados de evaluacion_generacion_detalle_raft.csv (evaluados antes, con modelos de respaldo permitidos)",
    "Evaluados en este notebook, sin modelos de respaldo, temperature = 0",
    "Promedio de los tres promedios por juez",
    "Promedio de los puntajes disponibles de la pregunta; n_jueces indica cuántos aportaron valor",
    str(RUTA_RESPUESTAS)]})

archivo = DIR_SALIDA / "faithfulness_raft_jueces.xlsx"
with pd.ExcelWriter(archivo) as xw:
    resumen.to_excel(xw, sheet_name="Resumen", index=False)
    detalle_xlsx.to_excel(xw, sheet_name="Detalle", index=False)
    puestos.reset_index().rename(columns={"index": "estilo_prompt"}).to_excel(xw, sheet_name="Puestos_por_juez", index=False)
    acuerdo.to_excel(xw, sheet_name="Acuerdo_entre_jueces", index=False)
    notas.to_excel(xw, sheet_name="Notas", index=False)
resumen.to_csv(DIR_SALIDA / "resumen.csv", index=False, encoding="utf-8-sig")
detalle_xlsx.to_csv(DIR_SALIDA / "detalle.csv", index=False, encoding="utf-8-sig")
print("Guardado en", archivo.resolve())